# Mushroom body extraction via atlas registration (2026_07_30_Lisa)

This dataset has **no MB-specific driver** -- ch0 is nc82 (pan-neuronal/synaptic neuropil,
whole brain) and ch1 is 5HT. Since there's no local intensity contrast to segment the MB
directly, this notebook instead **registers each scene's nc82 channel to a standard reference
brain (JFRC2) that already has the mushroom body outlined**, then warps that outline back onto
the scene -- following the strategy in Peng et al. 2011 (*Nat. Methods* 8:493-500, the
BrainAligner paper: nc82-based registration to a JFRC2-space target) and Jenett et al. 2012
(*Cell Reports* 2:991-1001, "Annotation" section: defining a volume-of-interest mask on the
standard brain and quantifying signal within it once registered -- Figure 7 shows MB as one of
their 68 standard regions).

**Atlas source**: `VirtualFlyBrain/DrosAdultBRAINdomains` (github.com/VirtualFlyBrain/DrosAdultBRAINdomains),
the standard Ito et al. 2014 (*Neuron* 81:755-765) neuropil nomenclature, distributed in the
same JFRC2 template space as Peng/Jenett used.

- `template/JFRCtemplate2010.nrrd` -- the reference brain (itself nc82-stained, so this is a
  same-modality registration -- the easiest case)
- `combinedIndexFiles/JFRCtempate2010.mask130819_Original.nrrd` -- integer-labeled region mask
- `refData/Original_Index.tsv` -- label ID -> region name lookup

**Registration engine**: `antspyx` (ANTsPy) -- affine + deformable (SyN), run on the nc82
channel only; the resulting transform is then applied to warp the MB label mask onto each
scene's native image space.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import platform
import numpy as np
import matplotlib.pyplot as plt
import ants
from aicsimageio import AICSImage

system = platform.system()
if system == 'Linux':
    home = '/home/gerard/'
elif system == 'Darwin':
    home = '/Users/gerard/'
elif system == 'Windows':
    home = 'C:/Users/cviko/'

try:
    sys.path.append(os.path.abspath(os.path.join(os.pardir, 'src')))
    from data_processing import describe_acquisition
except ImportError:
    path2add = home + 'analysis/confocal/src'
    sys.path.append(path2add)
    from data_processing import describe_acquisition

data_home = home + 'data/confocal/'
atlas_home = data_home + 'atlas_JFRC2/'


## 1. Load the atlas: template, region labels, and the MB label IDs

`Original_Index.tsv` splits the mushroom body into 4 sub-regions per hemisphere (pedunculus,
vertical lobe, medial lobe, calyx) -- combine all 8 label IDs into one binary "whole MB" mask.

In [ ]:
template = ants.image_read(atlas_home + 'JFRCtemplate2010.nrrd')
label_img = ants.image_read(atlas_home + 'JFRCtempate2010.mask130819_Original.nrrd')

print('template shape:', template.shape, ' spacing (um):', template.spacing)
print('label shape:', label_img.shape, ' spacing (um):', label_img.spacing)

# from refData/Original_Index.tsv: MB sub-regions, right (17,18,19,36) + left (64,65,66,81)
MB_LABEL_IDS = [17, 18, 19, 36, 64, 65, 66, 81]

label_arr = label_img.numpy()
mb_mask_arr = np.isin(label_arr, MB_LABEL_IDS).astype(np.float32)
mb_mask_template = ants.from_numpy(mb_mask_arr, origin=label_img.origin,
                                    spacing=label_img.spacing, direction=label_img.direction)

print(f'MB voxels in template: {int(mb_mask_arr.sum())} '
      f'({100 * mb_mask_arr.sum() / mb_mask_arr.size:.2f}% of volume)')

# kept around uncropped -- section 2b crops template/label_img/mb_mask_template in place
# once the scene's actual FOV size is known, replacing these names with cropped versions
template_full = template
label_img_full = label_img


## 2. Load one scene's nc82 channel (the moving image)

Starting with scene 0 (`NGS_20min`) as the test case before running all 4 scenes. Voxel
spacing must be set explicitly and correctly (in the same physical units, um) on the ANTs
image -- registration operates in physical space, not voxel indices, so this is what lets a
coarse-XY/fine-Z scene register correctly against a differently-sampled template.

In [ ]:
date = '2026_07_30'
user = 'Lisa'
lif_path = data_home + date + '_' + user + '/Project.lif'

info = describe_acquisition(lif_path, do_print=False)
scene_names = list(info.keys())
print(scene_names)

scene = 0
img = AICSImage(lif_path)
img.set_scene(img.scenes[scene])

vxy = info[scene_names[scene]]['voxel_xy_um']
vz = info[scene_names[scene]]['voxel_z_um']
print(f'scene {scene} ({scene_names[scene]}): vxy={vxy:.4f} um/px, vz={vz:.4f} um/step')

nc82_stack = img.get_image_data('ZYX', T=0, C=0).astype(np.float32)  # ch0 = nc82
print('nc82 stack shape (ZYX):', nc82_stack.shape)

moving = ants.from_numpy(nc82_stack, spacing=(vz, vxy, vxy))


## 2b. Crop the template to match this scene's actual field of view

The previous crop (see chat history) sized itself from the atlas's own central-brain label
bounding box, then squared that up -- but that has nothing to do with how big Lisa's actual
scene is, or where within the central brain it sits. Redone here the way it should have been
from the start: **crop size comes directly from the scene's own physical field of view**
(`shape_xy_px * vxy`, converted to template voxels via the template's own spacing), centered
on the template's X (left-right) axis, and positioned at the **top of the Y axis** rather than
centered -- per direct observation of the data, Lisa's crop only captures the upper part of
the brain in Y, not the full central-brain Y-extent. Z is left at full range since we don't
have an equivalent read on Z positioning yet.

If "top of Y" turns out to be the wrong end once you can actually see this rendered
correctly, flip `Y_ANCHOR` below to `'bottom'`.

In [ ]:
Y_ANCHOR = 'top'  # 'top' (y starts at 0) or 'bottom' (y ends at the last index)

fov_x_um = nc82_stack.shape[2] * vxy
fov_y_um = nc82_stack.shape[1] * vxy
print(f'scene FOV: {fov_x_um:.1f} x {fov_y_um:.1f} um')

crop_size_x_vox = int(round(fov_x_um / template_full.spacing[0]))
crop_size_y_vox = int(round(fov_y_um / template_full.spacing[1]))
print(f'-> template voxels needed: {crop_size_x_vox} x {crop_size_y_vox}')

x_center = template_full.shape[0] / 2
x_min = int(round(x_center - crop_size_x_vox / 2))
x_max = x_min + crop_size_x_vox

if Y_ANCHOR == 'top':
    y_min, y_max = 0, crop_size_y_vox
else:
    y_max = template_full.shape[1] - 1
    y_min = y_max - crop_size_y_vox

lowerind = (x_min, y_min, 0)
upperind = (x_max, y_max, template_full.shape[2] - 1)
print('crop lowerind:', lowerind, ' upperind:', upperind)

template = ants.crop_indices(template_full, lowerind, upperind)
label_img = ants.crop_indices(label_img_full, lowerind, upperind)
label_arr = label_img.numpy()
mb_mask_arr = np.isin(label_arr, MB_LABEL_IDS).astype(np.float32)
mb_mask_template = ants.from_numpy(mb_mask_arr, origin=label_img.origin,
                                    spacing=label_img.spacing, direction=label_img.direction)

print('cropped template shape:', template.shape, ' physical size (um):',
      np.array(template.shape) * np.array(template.spacing))
mb_total = np.isin(label_img_full.numpy(), MB_LABEL_IDS).sum()
print(f'MB voxels retained: {mb_mask_arr.sum():.0f} / {mb_total} '
      f'({100 * mb_mask_arr.sum() / mb_total:.1f}%)')

# Sanity-check render, correctly oriented (see 2c below for why axis order matters --
# this is a Z-max-projection, so it's not affected by the same napari default-slice bug,
# but slicing the WRONG axis here would still look wrong -- axis 2 is Z, confirmed in
# chat history from matching physical brain dimensions against Peng et al. 2011).
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(template.numpy().max(axis=2).T, cmap='gray')
ax.set_title(f'Cropped template (scene {scene} FOV, Y-anchor={Y_ANCHOR}), Z-max-projection')
ax.axis('off')
fig.tight_layout()


## 2c. Landmark-based initial alignment

Automatic orientation search (9 flip/rotation variants x 2 templates -- see chat history)
never found a confidently-good starting alignment: the best scores were weak and inconsistent,
and improved *less* once the template was cleaned up (cropped to remove the optic lobes),
which is the opposite of what a real correspondence should do. **By default, `ants.registration`
initializes from a naive center-of-mass alignment** (confirmed from its own docstring) -- for a
small, asymmetric central-brain-only crop, there's no guarantee that's anywhere near correct,
and pure intensity-based affine optimization has no way to escape a bad starting basin on its
own. This is exactly the problem Peng et al. 2011's BrainAligner solves with automatic landmark
detection instead of naive affine init -- here we do the same thing manually.

**Important axis-order note**: `template`'s own array is (X, Y, Z) -- napari's default
behavior for any 3D array is to treat **axis 0 as the scrollable slice dimension**. If we
handed it `template.numpy()` directly, axis 0 is X (not Z), so napari's slider would scroll
through *left-right position* and show you a thin front-to-back/depth cross-section at each
step -- not the recognizable top-down view (confirmed directly: that's exactly what produced
the "unrecognizable, not symmetric" image before this fix). Both images are transposed to
(Z, Y, X) before being added, so napari's default slider actually scrolls through depth for
both, and what you see when you first open each viewer is the recognizable, roughly
symmetric top-down brain view -- scrolling should only refine that, not completely change
what you're looking at.

**How to click points in napari**, step by step:
1. Each viewer opens with an image layer and an empty `landmarks` Points layer, already
   selected in the layer list (bottom-left panel).
2. In the layer controls (top-left panel), click the **"Add points"** tool (the icon that
   looks like a dot with a `+`, or press `2`) so clicks add points instead of panning.
3. Use the slider at the bottom of the viewer to scroll through Z (depth) and find a
   recognizable, distinct feature -- click it once to place a point.
4. Repeat for 5-8 total points, picking features spread across X, Y, *and* Z (not all in one
   slice) -- pick things you're genuinely confident are the same real structure in both
   viewers, not just similar-looking blobs.
5. **Click landmarks in both viewers in the same order** -- point 1 in the template must be
   the same real feature as point 1 in the scene, point 2 the same as point 2, etc.
6. Leave both viewers open (don't close them), then run the next cell.

In [ ]:
import napari

# Both images transposed to (Z, Y, X) before adding -- see markdown above for why napari's
# default axis-0-is-the-slider behavior needs this. nc82_stack is already (Z,Y,X) natively
# (aicsimageio's 'ZYX' request), so only the template (native X,Y,Z) actually needs it.
template_zyx = np.transpose(template.numpy(), (2, 1, 0))

viewer_template = napari.Viewer(title='TEMPLATE -- click landmarks here')
viewer_template.add_image(template_zyx, name='template', colormap='gray')
template_points_layer = viewer_template.add_points(name='landmarks', ndim=3, size=8, face_color='red')

viewer_scene = napari.Viewer(title=f'SCENE {scene} ({scene_names[scene]}) -- click landmarks here, SAME ORDER')
viewer_scene.add_image(nc82_stack, name='nc82', colormap='gray')
scene_points_layer = viewer_scene.add_points(name='landmarks', ndim=3, size=8, face_color='red')

print('Click corresponding landmarks in both viewers, same order, then run the next cell.')


Run this once you've clicked matching points in both viewers.

In [ ]:
template_points_idx_zyx = template_points_layer.data  # (N, 3), in the transposed (Z,Y,X) display order
scene_points_idx = scene_points_layer.data             # (N, 3), nc82_stack's own voxel indices (Z,Y,X)

assert len(template_points_idx_zyx) == len(scene_points_idx), (
    f'Point count mismatch: {len(template_points_idx_zyx)} template points vs '
    f'{len(scene_points_idx)} scene points -- click the same number, in the same order, in both.'
)
assert template_points_idx_zyx.shape[1] == 3 and scene_points_idx.shape[1] == 3, (
    f'Expected 3D points (N,3), got template shape {template_points_idx_zyx.shape} and '
    f'scene shape {scene_points_idx.shape} -- a Points layer created without ndim=3 silently '
    f'defaults to 2D even for a 3D image (this bit us once already: fixed by add_points(...,'
    f' ndim=3, ...) in the previous cell). If you still see this, re-run the viewers cell to '
    f'recreate the Points layers with the fix, then re-click.'
)
print(f'{len(template_points_idx_zyx)} landmark pairs')

# template's own ANTsImage indexing is still its native (X,Y,Z) -- convert the
# napari-display-order (Z,Y,X) clicks back before calling transform_index_to_physical_point.
template_points_idx = template_points_idx_zyx[:, [2, 1, 0]]

# transform_index_to_physical_point requires *integer* indices (napari's Points layer
# stores float sub-pixel click coordinates, so round before converting) -- passing floats
# fails to match any of its bound-function overloads.
template_points_phys = np.array([
    ants.transform_index_to_physical_point(template, [int(round(v)) for v in idx])
    for idx in template_points_idx
])
scene_points_phys = np.array([
    ants.transform_index_to_physical_point(moving, [int(round(v)) for v in idx])
    for idx in scene_points_idx
])

def _fit_similarity_allow_reflection(moving_pts, fixed_pts):
    """Umeyama similarity fit (rotation-or-reflection + one uniform scale + translation),
    deliberately WITHOUT the handedness correction ants.fit_transform_to_paired_points applies
    for transform_type='rigid'/'similarity'. That correction forces the fit to a proper
    rotation (det=+1) even when the best unconstrained orthogonal alignment between the two
    point sets is a reflection (det=-1) -- confirmed to be exactly the situation here via the
    Kabsch-determinant diagnostic below (came back -1.000, a clean reflection, not noise).
    Forcing a rotation onto a genuinely reflective correspondence distorts whichever single
    axis absorbs the correction -- this reproduced an isolated ~58% X-axis coverage failure
    with an otherwise-clean fit (Y/Z ~97-98%), while pairwise landmark distances (reflection-
    invariant) looked fine the whole time. Verified against a synthetic reflected point set:
    forcing det=+1 gave ~25-55 um residuals, allowing the reflection gave ~1e-14 um residuals.
    """
    center_fixed = fixed_pts.mean(axis=0)
    center_moving = moving_pts.mean(axis=0)
    x = fixed_pts - center_fixed
    y = moving_pts - center_moving
    C = y.T @ x
    U, D, Vt = np.linalg.svd(C)
    R = U @ Vt  # no det(R)<0 correction -- reflection allowed
    scale = D.sum() / (x ** 2).sum()
    A = scale * R
    translation = center_moving - A @ center_fixed
    return A, translation


A_fit, t_fit = _fit_similarity_allow_reflection(scene_points_phys, template_points_phys)
print(f'fitted linear part determinant: {np.linalg.det(A_fit):.3f} '
      f'({"reflection" if np.linalg.det(A_fit) < 0 else "proper rotation"} + uniform scale)')
landmark_transform = ants.create_ants_transform(matrix=A_fit, translation=t_fit, dimension=3)

landmark_transform_path = '/tmp/landmark_init.mat'
ants.write_transform(landmark_transform, landmark_transform_path)
print('saved landmark-derived initial transform to', landmark_transform_path)


### 2c-diagnostic. Sanity-check the fitted transform before trusting it

If the QC figure in section 4 shows the registered nc82 as a tiny fragment pinned to one
corner/edge of the template canvas, that's the signature of a **degenerate scale** in the
landmark-fitted affine, not a SyN problem -- SyN only refines locally from wherever the initial
transform puts things, it can't recover from the moving image being shrunk to a sliver.
This cell checks for that directly instead of guessing:

1. Compares each canvas's physical size (scene FOV vs template crop) -- these were deliberately
   made close in section 2b, so a registration starting from a sane place should already have
   the two images roughly overlapping in extent.
2. Compares pairwise distances between the same two landmarks in each point set -- if landmark
   pair (i, j) is e.g. 40 um apart in the template but 4 um apart in the scene, that pair is
   telling the fit two wildly different physical scales, and **also flags a stale
   `/tmp/landmark_init.mat` from a previous run** -- if section 2c errored out (e.g. the earlier
   `IndexError`) *after* writing a transform from a bad/partial click set, and the file on disk
   never got overwritten with a good one, the registration cell would silently reuse that old
   bad transform even after the click issue was fixed upstream.
3. Decomposes the fitted affine's linear part via SVD -- the singular values are exactly the
   scale factors along each principal axis. Close to 1.0 is expected (scene and template are
   physically similar-sized); anything near 0 or far from 1 means the fit collapsed the image.
4. **Directly checks canvas coverage, per axis** -- samples a grid of points across the
   template's physical bounding box, maps each through the fitted transform, and reports what
   fraction land inside the scene's actual valid data range, broken out per axis rather than
   as one combined number. This split matters: X/Y were deliberately sized to match the scene's
   FOV (section 2b), so low coverage there is a real problem. **Z was deliberately left at the
   template's full, uncropped range** (we don't yet have a read on where in Z the scene sits),
   so a substantial Z shortfall is *expected* even under a correct fit -- capped at roughly
   `(scene Z depth) / (template Z crop depth)` regardless of how good the transform is. A single
   combined 3D coverage number conflates these two very different situations, which is why it's
   reported per axis here instead.

   (Note: switching `'affine' -> 'similarity'` below removes the earlier 12-DOF fit's
   per-axis-independent scale/shear -- a real overfitting risk with only a handful of clicked
   points -- but empirically it did not meaningfully change the combined coverage number in this
   dataset, which is consistent with most of that shortfall being the expected Z effect rather
   than an X/Y problem the transform type could fix. The per-axis breakdown below is what
   actually distinguishes these, not the transform-type choice alone.)

In [ ]:
print('scene FOV (um):   ', np.array(nc82_stack.shape[::-1]) * np.array([vxy, vxy, vz]))
print('template crop (um):', np.array(template.shape) * np.array(template.spacing))

n_pts = len(template_points_phys)
print(f'\n{n_pts} landmark pairs -- pairwise physical distances (um):')
print(f'{"pair":>8} {"template":>10} {"scene":>10} {"ratio":>8}')
ratios = []
for i in range(n_pts):
    for j in range(i + 1, n_pts):
        d_t = np.linalg.norm(template_points_phys[i] - template_points_phys[j])
        d_s = np.linalg.norm(scene_points_phys[i] - scene_points_phys[j])
        ratio = d_t / d_s if d_s > 0 else np.nan
        ratios.append(ratio)
        print(f'{i}-{j:>6} {d_t:10.1f} {d_s:10.1f} {ratio:8.2f}')

ratios = np.array(ratios)
print(f'\nratio (template_dist / scene_dist): mean={np.nanmean(ratios):.2f}, '
      f'std={np.nanstd(ratios):.2f} -- should cluster near 1.0 for physically comparable '
      f'FOVs; large spread or a mean far from 1 means the clicked points themselves are '
      f'inconsistent, not just a code bug.')

# Reflection (mirror-flip) check -- pairwise distances are *invariant under reflection*, so
# the check above can look perfect even if the two point sets are actually mirror images of
# each other (e.g. a left-right flip between how the template and scene appear onscreen).
# The fit above (_fit_similarity_allow_reflection) already allows for this -- this block is
# now purely informational, confirming *why* a reflection-allowing fit was needed rather than
# flagging an unresolved problem. If this dataset's mirror relationship is consistent (likely,
# if it comes from a fixed acquisition/display convention rather than click luck), expect the
# same negative determinant on the other 3 scenes too -- not a new problem each time.
x_c = template_points_phys - template_points_phys.mean(axis=0)
y_c = scene_points_phys - scene_points_phys.mean(axis=0)
C = y_c.T @ x_c
U, _, Vt = np.linalg.svd(C)
kabsch_det = np.linalg.det(U @ Vt)
print(f'\nKabsch alignment determinant: {kabsch_det:.3f}')
if kabsch_det < 0:
    print('a reflection between the two point sets is expected and already handled by the '
          'reflection-allowing fit above -- this is informational, not a warning. (A forced '
          'proper-rotation fit -- e.g. ants.fit_transform_to_paired_points\'s own '
          '\'similarity\'/\'rigid\' -- would distort one axis to compensate instead; that\'s '
          'what produced the earlier isolated X-axis coverage failure.)')
else:
    print('no reflection -- the two point sets have consistent handedness; a rotation-only fit '
          'would have been fine too.')

# Affine parameters: first 9 = 3x3 linear part (row-major), last 3 = translation.
params = np.array(landmark_transform.parameters)
linear = params[:9].reshape(3, 3)
translation = params[9:]
singular_values = np.linalg.svd(linear, compute_uv=False)
print(f'\nfitted affine linear-part singular values (scale factors per axis): {singular_values}')
print(f'translation (um): {translation}')
if np.any(singular_values < 0.3) or np.any(singular_values > 3.0):
    print('\n*** WARNING: scale factor far from 1.0 -- this transform will shrink or blow up '
          'the moving image. Re-click landmarks (and double check ndim=3 / point order), or '
          'delete /tmp/landmark_init.mat if you suspect a stale file, then re-run from 2c. ***')

# Fit residual on the ACTUAL clicked points (not the earlier synthetic sanity check) --
# a similarity transform has 7 DOF (3 rotation + 1 scale + 3 translation) fit against 5 points
# (15 equations), so it's overdetermined and won't necessarily hit near-zero residual the way
# the synthetic 1-solution-exists test did. If residual here is small, the fit is honest and
# any remaining coverage shortfall is a genuine extrapolation/local-warp limitation (a global
# similarity transform can't perfectly explain every point when real anatomy isn't perfectly
# rigid) -- exactly what SyN's local deformation step exists to fix. If residual is large,
# something is still off with the points themselves (e.g. one mis-clicked pair dragging the
# whole fit), not just extrapolation.
pred_scene_phys = np.array([landmark_transform.apply_to_point(tuple(p)) for p in template_points_phys])
fit_residuals = np.linalg.norm(pred_scene_phys - scene_points_phys, axis=1)
print(f'\nfit residual per landmark pair (um): {fit_residuals}')
print(f'mean: {fit_residuals.mean():.1f} um, max: {fit_residuals.max():.1f} um '
      f'(scene FOV is ~234 um across in X/Y, for scale)')
if fit_residuals.max() > 20:
    worst = int(np.argmax(fit_residuals))
    print(f'*** point {worst} has the largest residual ({fit_residuals[worst]:.1f} um) -- '
          f'double check that click (same real feature, correct order) before trusting the fit. ***')

# Direct coverage check: does the fitted transform actually put the scene's data inside the
# template canvas? SVD scale factors alone can look individually plausible (all within a
# generous 0.3-3.0 band) while still leaving a large chunk of the canvas mapping to nothing --
# only checking where things actually land catches that.
#
# IMPORTANT: template's own array/physical axis order is (X, Y, Z); moving's is (Z, Y, X)
# (built from nc82_stack, aicsimageio's ZYX request, no reordering). landmark_transform's
# INPUT follows template's (X,Y,Z) convention and its OUTPUT follows moving's (Z,Y,X)
# convention -- a bare linear map doesn't care, but code printing per-axis results has to get
# this right, or it silently mislabels which physical axis it's reporting on. (This bit us
# once already: an earlier version of this cell printed axis_names=['X','Y','Z'] against the
# *output* array, which is actually (Z,Y,X)-ordered -- so its "X: 58%" was really reporting Z
# coverage, and its "Z: 99%" was really reporting X. Both orderings happen to agree only on
# the middle slot, Y, which is why Y alone looked right by coincidence. Rewritten below with
# explicit per-axis variables instead of parallel positional arrays, specifically to prevent
# this mistake from being reintroduced.)
template_extent_xyz = np.array(template.shape) * np.array(template.spacing)   # (X, Y, Z)
moving_extent_zyx = np.array(moving.shape) * np.array(moving.spacing)         # (Z, Y, X)
moving_lo_zyx = np.array(moving.origin)
moving_hi_zyx = moving_lo_zyx + moving_extent_zyx

grid_xyz = np.stack(np.meshgrid(
    np.linspace(0, template_extent_xyz[0], 15),
    np.linspace(0, template_extent_xyz[1], 15),
    np.linspace(0, template_extent_xyz[2], 15),
    indexing='ij',
), axis=-1).reshape(-1, 3) + np.array(template.origin)

mapped_zyx = np.array([landmark_transform.apply_to_point(tuple(p)) for p in grid_xyz])
inside_zyx = (mapped_zyx >= moving_lo_zyx) & (mapped_zyx <= moving_hi_zyx)
coverage = inside_zyx.all(axis=1).mean()

# Combined 3D coverage conflates two very different things: real misalignment (fixable by
# better points/transform) vs. an *expected* shortfall baked into section 2b's crop -- the
# template's Z range was deliberately left at full extent (we don't have a read on where the
# scene sits in Z yet), so even a perfect alignment can't cover more of the template's Z axis
# than (scene Z depth / template Z crop depth). Breaking coverage out per axis separates these:
# X/Y should be close to 100% (those axes were sized to match the scene), Z is expected to be
# well under 100% and that alone is not evidence of a bad fit.
print(f'\ntemplate-canvas coverage, per axis (fraction of that axis\'s grid points landing '
      f'inside the scene\'s valid range on that axis alone):')
z_ceiling = min(1.0, moving_extent_zyx[0] / template_extent_xyz[2])  # moving Z / template Z
print(f'  Z: {inside_zyx[:, 0].mean()*100:.0f}%  (expected ceiling from crop size alone: '
      f'~{z_ceiling*100:.0f}%)')
print(f'  Y: {inside_zyx[:, 1].mean()*100:.0f}%')
print(f'  X: {inside_zyx[:, 2].mean()*100:.0f}%')

print(f'\ncombined 3D coverage: {coverage*100:.0f}% of the template volume maps inside the '
      f"scene's valid data range under this transform")
if inside_zyx[:, 1].mean() < 0.85 or inside_zyx[:, 2].mean() < 0.85:
    print('*** WARNING: X or Y coverage is low -- those axes were sized to match the scene, so '
          'a low number there (unlike Z) is a real problem, not an expected crop artifact. '
          'Re-click with more/better-spread points before trusting SyN\'s output. ***')
else:
    print('X/Y coverage looks reasonable -- any shortfall is concentrated in Z, which is '
          'expected given the uncropped Z range (see markdown above), not a sign of a bad fit.')


## 3. Register: nc82 (moving) -> JFRC2 template (fixed)

`SyN` performs affine + deformable registration in one call, refining from the landmark-based
initial transform above instead of ants.registration's own default center-of-mass guess. This
can take a few minutes per scene at full resolution.

In [ ]:
registration = ants.registration(
    fixed=template, moving=moving,
    type_of_transform='SyN',
    initial_transform=[landmark_transform_path],
    verbose=False,
)
print('transforms:', registration['fwdtransforms'])
print('inverse transforms:', registration['invtransforms'])


## 4. QC: does the registered nc82 actually overlay the template?

Visual sanity check before trusting anything downstream -- same check Peng et al. 2011 describe
("we visually examined the transformed brains after global alignment"). If this doesn't look
right, warping the MB mask onto this scene isn't going to be meaningful either.

Top row: best-Z slice in template space (template / registered nc82 / overlay). Bottom row
adds **Z-max-projections**, including the **raw native nc82 before any registration** (its own
native coordinate grid, not template space) -- this is the one panel that's independent of
whatever the registration algorithm did, so it's the honest check for whether the raw data's
overall shape/orientation looks plausible relative to the template at all, rather than judging
orientation through the lens of a registration that might itself be wrong.

**Why "best-Z" and not "mid-Z"**: the template's Z crop was deliberately left at its full range
(section 2b -- we don't yet have a read on where in Z the scene sits), so the template's
geometric mid-Z can fall well outside the Z range the scene actually warps into (e.g. scene FOV
here is ~85 um deep vs a ~135 um template crop -- if the scene maps into one end of that range
rather than the center, slicing at the geometric middle shows mostly empty space with a thin
sliver of real signal at the edge -- which is exactly the "little piece... like a border" symptom).
Picking the slice with the most total signal in the *warped* volume is a data-driven fix for
this specific display issue -- it does not change the registration itself, only which slice
we're looking at.

In [ ]:
warped_moving = registration['warpedmovout']

template_np = template.numpy()
warped_np = warped_moving.numpy()

# NRRD axis order here is (X, Y, Z), not the (Z, Y, X) aicsimageio uses for the native
# scenes below -- template.shape = (1024, 512, 218); at 0.622 um isotropic spacing that's
# 637 x 318 x 136 um, matching the ~590 x 340 x 120 um adult fly brain dimensions in
# Peng et al. 2011 almost exactly, with the short (218) axis as Z/depth. So the top-down
# view used everywhere else in this notebook comes from slicing axis 2, not axis 0.
z_signal = warped_np.sum(axis=(0, 1))
z_mid = int(np.argmax(z_signal))

nonzero_z = np.where(z_signal > 0.01 * z_signal.max())[0]
z_span_um = (nonzero_z.max() - nonzero_z.min() + 1) * template.spacing[2] if len(nonzero_z) else 0
print(f'warped signal spans template Z indices {nonzero_z.min() if len(nonzero_z) else "-"}'
      f'-{nonzero_z.max() if len(nonzero_z) else "-"} of 0-{template_np.shape[2]-1} '
      f'(~{z_span_um:.0f} um of {template_np.shape[2] * template.spacing[2]:.0f} um total) -- '
      f'chosen best-Z slice: {z_mid}'
      + (' *** this is far from the template\'s geometric mid-Z -- confirms the mid-Z panel '
         'was showing an out-of-range slice, not a broken registration ***'
         if abs(z_mid - template_np.shape[2] // 2) > template_np.shape[2] // 6 else ''))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0, 0].imshow(template_np[:, :, z_mid].T, cmap='gray')
axes[0, 0].set_title(f'template (fixed), best-Z slice (z={z_mid})')
axes[0, 1].imshow(warped_np[:, :, z_mid].T, cmap='gray')
axes[0, 1].set_title(f'registered nc82, best-Z slice (z={z_mid})')
axes[0, 2].imshow(template_np[:, :, z_mid].T, cmap='Reds', alpha=0.6)
axes[0, 2].imshow(warped_np[:, :, z_mid].T, cmap='Greens', alpha=0.6)
axes[0, 2].set_title('overlay (template=red, nc82=green)')

axes[1, 0].imshow(template_np.max(axis=2).T, cmap='gray')
axes[1, 0].set_title('template, Z-max-projection')
axes[1, 1].imshow(nc82_stack.max(axis=0), cmap='gray')
axes[1, 1].set_title('RAW native nc82, Z-max-projection\n(native space, NOT registered)')
axes[1, 2].imshow(warped_np.max(axis=2).T, cmap='gray')
axes[1, 2].set_title('registered nc82, Z-max-projection\n(template space, post-registration)')

for ax in axes.ravel():
    ax.axis('off')
fig.tight_layout()


## 5. Warp the MB mask from template space onto this scene's native space

Uses the inverse transform (template -> subject direction) with nearest-neighbor
interpolation, since this is a label mask, not a continuous intensity image.

In [ ]:
mb_mask_native = ants.apply_transforms(
    fixed=moving, moving=mb_mask_template,
    transformlist=registration['invtransforms'],
    interpolator='nearestNeighbor',
)
mb_mask_native_arr = (mb_mask_native.numpy() > 0.5)

print(f'MB voxels in native scene space: {mb_mask_native_arr.sum()} '
      f'({100 * mb_mask_native_arr.sum() / mb_mask_native_arr.size:.2f}% of volume)')


## 6. QC: overlay the warped MB mask on the native nc82 stack

The real test -- does the warped MB outline actually sit on top of what looks anatomically
like the mushroom body in the original image?

In [ ]:
from matplotlib.colors import ListedColormap

z_native = nc82_stack.shape[0] // 2
mb_overlay_cmap = ListedColormap(['none', 'red'])

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(nc82_stack[z_native], cmap='gray')
axes[0].set_title(f'nc82, native space (z={z_native})')

axes[1].imshow(nc82_stack[z_native], cmap='gray')
axes[1].imshow(mb_mask_native_arr[z_native], cmap=mb_overlay_cmap, alpha=0.5)
axes[1].set_title('nc82 + warped MB mask (red)')
for ax in axes:
    ax.axis('off')
fig.tight_layout()

# Max-projection version -- a single Z-slice only catches whichever part of each
# hemisphere's MB happens to sit at that exact depth (real brains aren't perfectly
# flat/symmetric), so this is the more reliable check that *both* hemispheres warped
# to sensible, bounded locations rather than one being missed or smeared.
nc82_proj = nc82_stack.max(axis=0)
mb_proj = mb_mask_native_arr.max(axis=0)

fig2, ax2 = plt.subplots(figsize=(6, 6))
ax2.imshow(nc82_proj, cmap='gray')
ax2.imshow(mb_proj, cmap=mb_overlay_cmap, alpha=0.5)
ax2.set_title('nc82 Z-max-projection + warped MB mask (red)')
ax2.axis('off')
fig2.tight_layout()


## 6b. Frame-by-frame QC in napari

The static figures above only show one Z-slice or a flattened projection. `nc82_stack` and
`mb_mask_native_arr` are already the same shape and the same native (Z, Y, X) axis order (the
mask was warped with `fixed=moving`, so it's resampled directly onto the scene's own grid --
no transpose needed here, unlike the template/scene landmark viewers earlier), so they can go
straight into one viewer and be scrolled through together, frame by frame.

In [ ]:
import napari

viewer_mb_qc = napari.Viewer(title=f'MB mask QC -- scene {scene} ({scene_names[scene]})')
viewer_mb_qc.add_image(nc82_stack, name='nc82', colormap='gray')
viewer_mb_qc.add_labels(mb_mask_native_arr.astype(np.uint8), name='MB mask (warped)', opacity=0.5)
print('Scroll the Z slider at the bottom -- both layers share the same axis, so this checks '
      'colocalization frame by frame directly, more informative than the static figures above.')


## 6c. Would more landmark points improve this?

Likely yes, but not for the reason it might seem. The current fit (5 points, `similarity`
transform: rotation/reflection + one uniform scale + translation, 7 DOF) already has a small
residual on the points themselves (~4 um mean against a ~234 um canvas) -- so it's not
underfitting in an obvious way. But a similarity transform is **globally rigid**: it can't
locally warp one region differently from another. SyN's deformable step does that refinement,
but only within the neighborhood it's initialized into -- so accuracy specifically *at the MB*
depends on how well the 5 points happen to constrain that particular neighborhood, not just the
brain as a whole. If none of the 5 points were clicked on or near the MB itself, the global
fit could be excellent overall while still being a few voxels off exactly where it matters most.

**More useful than "more points" in general: points specifically bracketing the MB** (e.g. the
tips of the vertical and medial lobes, the calyx boundary) -- this tightens the fit exactly
where accuracy matters, and gives SyN a better (smaller-deformation) starting point right at
the region of interest rather than relying on it to correct a larger local gap on its own.
5-8 points spread across the *whole* brain (current approach) is reasonable for a first
attempt; if the QC above shows the mask consistently offset near the MB specifically, add 2-3
more points right at MB landmarks rather than scattering more everywhere.

In [ ]:
# From Original_Index.tsv, the 4 MB sub-structures per hemisphere are: pedunculus (axon
# tract), vertical lobe + medial lobe (the "lobe system" proper -- most likely what's meant by
# "lobules"), and calyx (dendritic input region). MB_LABEL_IDS above includes all 4; this
# isolates just the two lobes, excluding the peduncle tract and the calyx.
MB_LOBE_LABEL_IDS = [18, 19, 65, 66]  # MB_VL_R, MB_ML_R, MB_VL_L, MB_ML_L

mb_lobe_mask_arr = np.isin(label_arr, MB_LOBE_LABEL_IDS).astype(np.float32)
mb_lobe_mask_template = ants.from_numpy(mb_lobe_mask_arr, origin=label_img.origin,
                                         spacing=label_img.spacing, direction=label_img.direction)

mb_lobe_mask_native = ants.apply_transforms(
    fixed=moving, moving=mb_lobe_mask_template,
    transformlist=registration['invtransforms'],
    interpolator='nearestNeighbor',
)
mb_lobe_mask_native_arr = (mb_lobe_mask_native.numpy() > 0.5)
print(f'MB-lobe voxels in native scene space: {mb_lobe_mask_native_arr.sum()} '
      f'({100 * mb_lobe_mask_native_arr.sum() / mb_lobe_mask_native_arr.size:.2f}% of volume)')

# Compare against the full MB mask in the same napari viewer as a second Labels layer --
# swap in mb_lobe_mask_native_arr.astype(np.uint8) below and re-run, or add it alongside:
# viewer_mb_qc.add_labels(mb_lobe_mask_native_arr.astype(np.uint8), name='MB lobes only',
#                          opacity=0.5, color={1: 'cyan'})


## 7. Wrap into a function and run across all 4 scenes

Once the QC above looks right for scene 0, package the same steps into a function and run it
for every scene, saving each scene's MB mask (as a `.tif`, same shape as the native stack) for
downstream use (e.g. restricting 5HT quantification to the MB).

In [ ]:
from tifffile import imwrite

def register_and_extract_mb(lif_path, scene, info, template, mb_mask_template, channel=0):
    scene_names = list(info.keys())
    img = AICSImage(lif_path)
    img.set_scene(img.scenes[scene])

    vxy = info[scene_names[scene]]['voxel_xy_um']
    vz = info[scene_names[scene]]['voxel_z_um']
    nc82_stack = img.get_image_data('ZYX', T=0, C=channel).astype(np.float32)
    moving = ants.from_numpy(nc82_stack, spacing=(vz, vxy, vxy))

    reg = ants.registration(fixed=template, moving=moving, type_of_transform='SyN', verbose=False)
    mb_native = ants.apply_transforms(
        fixed=moving, moving=mb_mask_template,
        transformlist=reg['invtransforms'], interpolator='nearestNeighbor',
    )
    mb_mask_arr = (mb_native.numpy() > 0.5).astype(np.uint8)
    return nc82_stack, mb_mask_arr, reg


# results = {}
# for scene in range(len(scene_names)):
#     print(f'--- scene {scene}: {scene_names[scene]} ---')
#     nc82_stack, mb_mask_arr, reg = register_and_extract_mb(lif_path, scene, info, template, mb_mask_template)
#     results[scene] = {'nc82': nc82_stack, 'mb_mask': mb_mask_arr}
#
#     out_dir = data_home + date + '_' + user + f'/series_{scene}/masks/'
#     os.makedirs(out_dir, exist_ok=True)
#     imwrite(out_dir + 'msk_MB_registered.tif', mb_mask_arr, imagej=True)
#     print(f'saved MB mask: {mb_mask_arr.sum()} voxels')
